#1. What is Machine Learning? Explain the main types of Machine Learning with examples.

Machine Learning (ML) is a branch of artificial intelligence in which systems learn patterns from data to make predictions or decisions without being explicitly programmed for every case.
- Supervised learning: learns from labelled examples. Examples: predicting house prices (regression), detecting spam (classification).
- Unsupervised learning: finds structure in unlabelled data. Examples: customer segmentation using clustering, grouping similar documents.
- Reinforcement learning: an agent learns by taking actions and receiving rewards or penalties. Examples: game-playing agents and robot navigation.

#2. What is Supervised Learning? Explain the difference between Regression and Classification.

Supervised learning trains a model using input features paired with known target labels.
- Regression predicts a continuous numeric value, such as a house price or temperature.
- Classification predicts a discrete class/category, such as spam/not spam or disease/no disease.

#3. What is Linear Regression? Write the Linear Regression equation and explain intercept and coefficient.

Linear Regression models a numeric target as a linear combination of input features:

\[
\hat{y}=\beta_0+\beta_1x_1+\beta_2x_2+\cdots+\beta_px_p
\]

Here, \(\hat{y}\) is the predicted target, \(\beta_0\) is the intercept (predicted target when all features equal zero), and \(\beta_i\) is the coefficient for feature \(x_i\). A coefficient describes the expected change in prediction for a one-unit increase in that feature, holding the other features constant.

#4. Explain the following evaluation metrics used in Linear Regression: SSE, MSE, RMSE, R² Score, Adjusted R²

- SSE (Sum of Squared Errors): \(\sum_i(y_i-\hat{y}_i)^2\); total squared prediction error. Lower is better.
- MSE (Mean Squared Error): \(\frac{1}{n}\sum_i(y_i-\hat{y}_i)^2\); average squared error. Lower is better.
- RMSE (Root Mean Squared Error): \(\sqrt{MSE}\); error measured in the target's original units. Lower is better.
- R² score: \(1-\frac{SSE}{SST}\); proportion of target variance explained by the model. A value closer to 1 generally indicates a better fit; it can be negative on test data.
- Adjusted R²: adjusts R² for the number of predictors and sample size. It can decrease when unhelpful predictors are added. On a test set, this notebook uses \(1-(1-R^2)\frac{n-1}{n-p-1}\), where \(n\) is the number of test observations and \(p\) is the number of predictors.

#5. Explain the Machine Learning workflow steps from data collection to model evaluation.

1. Define the problem and target.
2. Collect and load data.
3. Inspect, clean, and prepare data (types, missing values, duplicates, outliers as appropriate).
4. Explore data using descriptive statistics and visualizations.
5. Select features and target.
6. Split data into training and testing sets.
7. Train a suitable model using training data.
8. Predict on unseen test data.
9. Evaluate using suitable metrics and inspect errors.
10. Interpret results, improve the model if needed, and communicate findings.

#6. Load the Boston Housing dataset and perform basic data exploration using: head() info() describe()


```
# If running in Google Colab, upload BostonHousing.csv when prompted.
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from google.colab import files
    if not os.path.exists("BostonHousing.csv"):
        uploaded = files.upload()
except ImportError:
    pass

df = pd.read_csv("BostonHousing.csv")
print("Dataset shape:", df.shape)
display(df.head())
```

#7. Perform Exploratory Data Analysis: Create pairplot Create correlation heatmap  Identify which features are highly correlated with house price (medv)

```
# Pairplot of target and selected features
pairplot_cols = [c for c in ["medv", "rm", "lstat", "ptratio", "crim"] if c in df.columns]
sns.pairplot(df[pairplot_cols], diag_kind="hist", corner=True)
plt.suptitle("Pairplot: Boston Housing Selected Features", y=1.02)
plt.show()

# Correlation heatmap
plt.figure(figsize=(12, 9))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap — Boston Housing")
plt.tight_layout()
plt.show()

# Correlation of each feature with medv
medv_corr = df.corr(numeric_only=True)["medv"].drop("medv").sort_values(ascending=False)
display(medv_corr.to_frame("correlation_with_medv"))

print("Strongest positive correlation:", medv_corr.idxmax(), round(medv_corr.max(), 3))
print("Strongest negative correlation:", medv_corr.idxmin(), round(medv_corr.min(), 3))
```

#8. Select independent variables (features) and dependent variable (target).Target variable: medv (house price)

```
X = df.drop(columns=["medv"])
y = df["medv"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Features:", X.columns.tolist())
```

#9. Split the dataset into training and testing sets (80% training, 20% testing).

```
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)
print("Training target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)
```

#10. Train a Linear Regression model and perform the following:*   Make predictions*   Calculate MAE *   Calculate MSE *   Calculate RMSE *   Calculate R² Score *   Calculate Adjusted R² *   Write the Linear Regression equation *   Interpret the model coefficients
```
# Train the model
model = LinearRegression()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
sse = np.sum((y_test - y_pred) ** 2)
r2 = r2_score(y_test, y_pred)

n = len(y_test)       # number of test observations
p = X_test.shape[1]   # number of predictors
adjusted_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

metrics = pd.DataFrame({
    "Metric": ["SSE", "MAE", "MSE", "RMSE", "R² Score", "Adjusted R²"],
    "Value": [sse, mae, mse, rmse, r2, adjusted_r2]
})
display(metrics.round(4))
```

```
# Linear regression equation
print(f"medv = {model.intercept_:.4f}", end="")
for feature, coef in zip(X.columns, model.coef_):
    sign = "+" if coef >= 0 else "-"
    print(f" {sign} {abs(coef):.4f}*{feature}", end="")
print()

# Coefficient table
coef_table = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
})
coef_table["Direction"] = np.where(coef_table["Coefficient"] >= 0, "Positive", "Negative")
display(coef_table.sort_values("Coefficient", key=lambda s: s.abs(), ascending=False).round(4))

print(f"Intercept: {model.intercept_:.4f}")
print("Interpretation: each coefficient is the predicted change in medv for a one-unit increase in that feature, holding all other features constant.")
```